# HyDRA V5 — Missing Experiments: V5-A (corrected) and V5-C (corrected)

**Objective:** Close the V5 matrix with the two runs that suffered RadiusCollapse
due to missing `radial_proj_min_r=0.5` guard.

| Variant | Ch1 | Ch2 | Previous result | This run |
|---------|-----|-----|-----------------|----------|
| **V5-A** | ✅ RiemannianAdamW | ❌ | RadiusCollapse (r_min not active) | Re-run with r_min=0.5 |
| **V5-C** | ✅ RiemannianAdamW | ✅ AngularLMHead | RadiusCollapse (r_min not active) | Re-run with r_min=0.5 |

**V5-B result for reference:** rdc\*=0.96 (Ch2 alone reduces DegEq by 91%)

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → Cell V5-AC

**Key fix confirmed active:** `[preflight] r_min guard: 0.5 ✅` must appear before training starts.

In [1]:
"""
Cell 1 — Environment Setup
==========================
Installs dependencies, optionally mounts Google Drive, creates the output
directory tree, and sets global determinism seeds.

Storage strategy (automatic):
  • Google Drive available  → PAPER_ROOT = /content/drive/MyDrive/HydraPaper
  • Drive unavailable/skipped → PAPER_ROOT = /content/HydraPaper
    (persists for the Colab session; download artifacts via Cell 12 ZIP export)

Directory structure:
    /checkpoints/   model snapshots (.pt)
    /logs/          raw training logs (.csv)
    /results/       derived metrics
    /figures/       publication figures (PNG 300 DPI + PDF vector)
    /tables/        LaTeX tables (booktabs format)
    /configs/       JSON reproducibility manifests
    /exports/       final ZIP archive
"""
import subprocess, sys, os

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'[WARN] {cmd!r} → {r.stderr[:200]}')

print('Installing dependencies...')
_run('pip install torch tqdm transformers accelerate datasets POT --quiet')
print('✅ Dependencies installed')

# ── Google Drive (optional) ────────────────────────────────
from pathlib import Path

USE_DRIVE = True   # FIX: try Drive first; falls back to local if unavailable
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PAPER_ROOT = Path('/content/drive/MyDrive/HydraPaper_VariantF').resolve()
    USE_DRIVE  = True
    print('✅ Google Drive mounted')
except Exception as _drive_err:
    PAPER_ROOT = Path('/content/HydraPaper_VariantF')
    print(f'⚠️  Drive unavailable ({type(_drive_err).__name__}: {_drive_err})')
    print(f'   → Using local storage: {PAPER_ROOT}')
    print('   Artifacts will be in the session ZIP export (Cell 12).')

SUBDIRS = ['checkpoints','logs','results','figures','tables','configs','exports']
for d in SUBDIRS:
    (PAPER_ROOT / d).mkdir(parents=True, exist_ok=True)
print(f'✅ Output root: {PAPER_ROOT}  (drive={USE_DRIVE})')

# ── Determinism ────────────────────────────────────────────
import torch, random, numpy as np
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic    = True
torch.backends.cudnn.benchmark        = False
torch.backends.cuda.matmul.allow_tf32 = False   # precision guard for Lorentz ops
torch.backends.cudnn.allow_tf32       = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Seeds set (seed={SEED})  device={DEVICE}  torch={torch.__version__}')
print(f'📂 PAPER_ROOT absoluto: {PAPER_ROOT}')
os.chdir(PAPER_ROOT)
print(f'📍 Working dir: {Path.cwd()}')

Installing dependencies...
✅ Dependencies installed
Mounted at /content/drive
✅ Google Drive mounted
✅ Output root: /content/drive/MyDrive/HydraPaper_VariantF  (drive=True)
✅ Seeds set (seed=42)  device=cuda  torch=2.10.0+cu128
📂 PAPER_ROOT absoluto: /content/drive/MyDrive/HydraPaper_VariantF
📍 Working dir: /content/drive/MyDrive/HydraPaper_VariantF


In [2]:
"""
Cell 2 — Source Upload & Mount  [HARDENED: cache-purge + version lock]
=======================================================================
Purges any cached cgt installation, extracts the uploaded zip fresh,
then PROVES which code version is active before allowing execution to
continue.  If target_radius != 1.5, execution STOPS here.
"""
import importlib, sys, os, zipfile

# ── STEP 0: Hard purge of all cached cgt modules ───────────
# Colab may hold stale compiled .pyc or already-imported modules in memory.
# Remove them ALL before touching the file system.
_stale = [k for k in sys.modules if k == 'cgt' or k.startswith('cgt.')]
for _k in _stale:
    del sys.modules[_k]
print(f"Purged {len(_stale)} cached cgt module(s) from sys.modules")

# ── STEP 1: Wipe previous extraction on disk ───────────────
import shutil
for _old in ['/content/src', '/content/cgt']:
    if os.path.exists(_old):
        shutil.rmtree(_old)
        print(f"Deleted old directory: {_old}")

# ── STEP 2: Upload and extract ─────────────────────────────
from google.colab import files as _colab_files
print('Upload src_FULL_V5.zip when the dialog appears...')
_uploaded = _colab_files.upload(target_dir='/tmp')
if not _uploaded:
    raise RuntimeError('No file uploaded.')
_zip_name = list(_uploaded.keys())[0]
_zip_path = f'/{_zip_name}'

os.makedirs('/content/src', exist_ok=True)
with zipfile.ZipFile(_zip_path, 'r') as _zf:
    _zf.extractall('/content/src')
print(f'Extracted: {_zip_name}')

# ── STEP 3: Mount sys.path ─────────────────────────────────
_candidates = ['/content/src', '/content/src/src', '/content']
_cgt_root = next(
    (p for p in _candidates if os.path.isdir(os.path.join(p, 'cgt'))), None
)
if _cgt_root is None:
    raise RuntimeError('cgt/ not found in extracted zip. Wrong zip uploaded?')
if _cgt_root not in sys.path:
    sys.path.insert(0, _cgt_root)

# ── STEP 4: Grep the actual file on disk (mandatory) ───────
import subprocess
_distill_path = os.path.join(_cgt_root, 'cgt', 'distillation', 'distillation_v2.py')
if not os.path.exists(_distill_path):
    raise FileNotFoundError(f'distillation_v2.py not found at {_distill_path}')

_grep = subprocess.run(
    ['grep', '-n', 'target_radius', _distill_path],
    capture_output=True, text=True
)
print("\n[DISK VERIFY] grep target_radius distillation_v2.py:")
print(_grep.stdout.strip())

# ── STEP 5: Parse the exact value from file ────────────────
_target_on_disk = None
for _line in _grep.stdout.splitlines():
    if '= ' in _line and 'target_radius' in _line and '#' not in _line.split('target_radius')[0]:
        try:
            _target_on_disk = float(_line.split('=')[1].split('#')[0].strip())
            break
        except ValueError:
            pass

print(f"[DISK VERIFY] target_radius on disk = {_target_on_disk}")

# ── STEP 6: FAIL-FAST if wrong version ─────────────────────
if _target_on_disk is None:
    raise RuntimeError(
        "ABORT: Could not parse target_radius from distillation_v2.py. "
        "Inspect the file manually."
    )
if _target_on_disk >= 2.0:
    raise RuntimeError(
        f"ABORT: target_radius={_target_on_disk} on disk — OLD code loaded. "
        f"Upload src_FULL_V4.zip (not src__7_.zip or any original version). "
        f"Expected: target_radius=1.5"
    )

print(f"\n✅ DISK VERSION CONFIRMED: target_radius={_target_on_disk}")

# ── STEP 7: Import and runtime confirmation ────────────────
import cgt
from cgt.distillation.distillation_v2 import TeacherDistillationLossV2
import inspect, re

_src = inspect.getsource(TeacherDistillationLossV2._radius_loss)
_m = re.search(r'target_radius\s*=\s*([\d.]+)', _src)
_runtime_val = float(_m.group(1)) if _m else None

print(f"[RUNTIME VERIFY] target_radius in loaded class = {_runtime_val}")

if _runtime_val is None or _runtime_val >= 2.0:
    raise RuntimeError(
        f"ABORT: Runtime target_radius={_runtime_val} — module reload failed. "
        f"Kernel may be caching a stale version. "
        f"Go to Runtime → Restart runtime, then re-run from Cell 1."
    )

print(f"✅ RUNTIME VERSION CONFIRMED: target_radius={_runtime_val}")
print(f"✅ cgt mounted  root={_cgt_root}  version={cgt.__version__}")

# ── STEP 8: Verify V4 new modules are present ──────────────
_v4_checks = {
    'HyperbolicProjectorV3': 'cgt.distillation.hyperbolic_projector',
    'GeodesicLMHeadV2':      'cgt.models.geodesic_lm_head',
    'RiemannianAdamW':       'cgt.dynamics.riemannian_adamw',
}
_v4_missing = []
for _cls, _mod in _v4_checks.items():
    try:
        _m = importlib.import_module(_mod)
        getattr(_m, _cls)
        print(f"  ✅ {_cls} present in {_mod}")
    except Exception as _e:
        _v4_missing.append(f"{_cls}: {_e}")
        print(f"  ⚠️  {_cls} missing — {_e}")

if _v4_missing:
    print(f"⚠️  V4 modules missing (non-critical, training still works): {_v4_missing}")
else:
    print("✅ src_FULL_V4.zip confirmed — all new modules present")
print("\n" + "="*60)
print("V4 VERIFIED: target_radius=1.5 + new modules ACTIVE. Safe to proceed.")
print("="*60)

Purged 0 cached cgt module(s) from sys.modules
Deleted old directory: /content/src
Upload src_FULL_V5.zip when the dialog appears...


Saving src.zip to /tmp/src (1).zip
Extracted: /tmp/src (1).zip

[DISK VERIFY] grep target_radius distillation_v2.py:
942:        # FIX: target_radius 4.0 → 1.5
961:        target_radius = 1.5
962:        return F.mse_loss(radius.clamp(max=4.0), torch.full_like(radius, target_radius))
[DISK VERIFY] target_radius on disk = 1.5

✅ DISK VERSION CONFIRMED: target_radius=1.5
[RUNTIME VERIFY] target_radius in loaded class = 1.5
✅ RUNTIME VERSION CONFIRMED: target_radius=1.5
✅ cgt mounted  root=/content/src/src  version=1.0.0
  ✅ HyperbolicProjectorV3 present in cgt.distillation.hyperbolic_projector
  ✅ GeodesicLMHeadV2 present in cgt.models.geodesic_lm_head
  ✅ RiemannianAdamW present in cgt.dynamics.riemannian_adamw
✅ src_FULL_V4.zip confirmed — all new modules present

V4 VERIFIED: target_radius=1.5 + new modules ACTIVE. Safe to proceed.


In [3]:
"""
Cell 3 — Unified Experiment Configuration
==========================================
Single source of truth for all hyperparameters.  Saving this block as JSON
to PAPER_ROOT/configs/ is sufficient to reproduce the entire run.

Model architecture (SafeHyperbolicModel)
-----------------------------------------
  vocab_size      : 50257  (GPT-2 BPE)
  n_embd          : 128    (hyperbolic intrinsic dim = ambient - 1)
  n_layer         : 4
  n_head          : 4
  n_positions     : 128    (sequence length)
  use_dynamics    : False during training; wrapper attached post-training

Dynamics (DynamicSLMWrapperV2 — attached post-training)
---------------------------------------------------------
  coupling_strength : 0.1  (over-sync fix from v2; was 1.0)
  dt                : 0.02 (ODE stability; was 0.05)
  num_steps         : 5    (phase collapse prevention; was 10)

Distillation (DistillationConfigV2)
------------------------------------
  lambda_distill : 0.3   teacher KL weight
  lambda_hidden  : 0.15  learnable projection alignment
  lambda_radius  : 0.05  geodesic radius regularisation
  lambda_contrast: 0.05  in-batch InfoNCE contrastive
  temperature    : 3.0   KL softening
  max_steps      : 10000

EarlyStoppingV3 (dual-EMA regime-sensitive stopping)
-----------------------------------------------------
  patience       : 10
  ema_beta       : 0.9   slow EMA (phase/display)
  ema_beta_fast  : 0.3   fast EMA (stopping decisions)
  min_delta      : 0.005 relative improvement threshold
  noise_mult     : 2.0   improvement must exceed 2×noise to hold patience
  plateau_thresh : 5     global stagnation confirmation
"""
import json
from pathlib import Path

# PAPER_ROOT is inherited from Cell 1 (Drive or local fallback)
CKPT_DIR   = PAPER_ROOT / 'checkpoints'
SEQ_LEN    = 128
SEED       = 42

# ── Run identification ─────────────────────────────────────
# Override RUN_SEED / RUN_TAG before running this cell to switch experiment.
#
#   RUN_SEED = 42   → seed_0  (Variant F default)
#   RUN_SEED = 1    → seed_1  (multi-seed validation)
#   RUN_SEED = 2    → seed_2  (multi-seed validation)
#
#   RUN_TAG = 'variant_f'  → hyperbolic Variant F (default)
#   RUN_TAG = 'euclidean'  → Euclidean baseline  (set before Cell 6d)
#   RUN_TAG = 'seed1'      → Variant F seed 1
#   RUN_TAG = 'seed2'      → Variant F seed 2
#
# Cell 6 reads RUN_TAG to isolate logs + checkpoints; no other cells change.
RUN_SEED = SEED          # override here: RUN_SEED = 1
RUN_TAG  = 'variant_f'   # override here: RUN_TAG  = 'euclidean'

# Deterministic seeding — applied immediately so dataset shuffles are reproducible
import random as _rnd, numpy as _np, torch as _tc
_tc.manual_seed(RUN_SEED)
_rnd.seed(RUN_SEED)
_np.random.seed(RUN_SEED)
if _tc.cuda.is_available():
    _tc.cuda.manual_seed_all(RUN_SEED)
import os as _os
_os.environ['PYTHONHASHSEED'] = str(RUN_SEED)  # Python dict/set ordering
print(f'Seeded: RUN_SEED={RUN_SEED}  RUN_TAG={RUN_TAG!r}')

CFG = {
    'model': {
        'vocab_size': 50257, 'n_embd': 128, 'n_layer': 4, 'n_head': 4,
        'n_positions': SEQ_LEN, 'hyperbolic_dim': 128,
        'initial_curvature': 1.0, 'learnable_curvature': False,
        'tie_word_embeddings': True, 'use_dynamics': False,
    },
    'dynamics': {
        'coupling_strength': 0.1, 'dt': 0.02, 'num_steps': 5,
        'use_hyperbolic_coupling': False, 'record_trajectory': True,
        'learnable_frequencies': True,
    },
    'training': {
        'alpha': 0.25, # reduced KL pressure — delays DegEq onset, more angular learning
        'temperature': 1.2,  # BALANCED: T=1.0 too aggressive → KL killed contrast diversity
        'lambda_distill': 0.30, # BALANCED: 0.35 overcrowded KL, restoring ratio
        'lambda_hidden': 0.15,
        'lambda_radius': 0.0,   # PPL-A: dead (MSE=0 always), freed budget
        'lambda_contrast': 0.1,  # FIX: 0.05→0.1 (contrastive repaired in src)
        'label_smoothing': 0.1, 'learning_rate': 3e-4,
        'weight_decay': 0.01, 'max_steps': 100000,
        'warmup_steps': 300, 'gradient_clip': 1.0,
        'eval_every': 200, 'log_every': 50, 'checkpoint_every': 500,
        'lr_floor': 0.05,  # lr_squeeze_after auto = 20% of max_steps (DO NOT hardcode)
        'deg_eq_action': 'none',              # overridden by Cell 6c
        'deg_eq_consecutive_required': 3,     # confirmations before action fires
        # ── Riemannian natural gradient (Amari 1998) ───────────────
        'riemannian_correct_vocab':   True,   # lm_head tangent (validated)
        'riemannian_correct_embed':   True,   # token embeddings
        'riemannian_correct_encoder': True,   # full encoder (Variant F)
        # ── Checkpoint resume ──────────────────────────────────────
        'resume_from_checkpoint':     True,   # auto-resume if checkpoint exists

        'teacher_hidden_dim': 768,
        'batch_size': 16, 'dataset': 'wikitext-2-raw-v1',
    },
    'early_stopping': {
        'class': 'EarlyStoppingV3',
        'patience': 25, 'ema_beta': 0.9, 'ema_beta_fast': 0.3,
        'min_delta': 0.003, 'window_size': 8, 'noise_mult': 1.5,
        'min_steps': 5000, 'plateau_threshold': 20,
        'logit_std_delta': 0.1, 'phase3_patience_multiplier': 1.5,
    },
    'reproducibility': {'seed': RUN_SEED, 'tf32_disabled': True},
}

config_path = PAPER_ROOT / 'configs' / 'hydra_v2_config.json'
with open(config_path, 'w') as f:
    json.dump(CFG, f, indent=2)

print('✅ Configuration saved')
print(f'   {config_path}')
print(f'   model      : {CFG["model"]["n_layer"]}L × {CFG["model"]["n_embd"]}d × {CFG["model"]["n_head"]}H')
print(f'   max_steps  : {CFG["training"]["max_steps"]}')
print(f'   early_stop : {CFG["early_stopping"]["class"]}  patience={CFG["early_stopping"]["patience"]}')

Seeded: RUN_SEED=42  RUN_TAG='variant_f'
✅ Configuration saved
   /content/drive/MyDrive/HydraPaper_VariantF/configs/hydra_v2_config.json
   model      : 4L × 128d × 4H
   max_steps  : 100000
   early_stop : EarlyStoppingV3  patience=25


In [4]:
"""
Cell 4 — Teacher and Student Model Setup
==========================================
Constructs:

GPT2TeacherWrapperV2
    Frozen GPT-2-small (117M parameters) as the distillation teacher.
    Parameters are frozen; only used for forward passes to produce
    soft targets and hidden-state alignment signals.

SafeHyperbolicModel (student)
    12M-parameter hyperbolic transformer operating on the Lorentz
    hyperboloid H^128 with curvature K=1 (learnable=False).
    Architecture:  4 layers × 128-dim × 4 heads × 512 FFN.
    LM head:       IntrinsicLorentz — projects from ambient R^129
                   to vocabulary logits via log_map_zero → linear.

Assertions verify that the patched LM head (v3) is present:
    lm_head.logit_scale  — learned temperature scaling
    lm_head.ambient_dim  — must equal n_embd + 1 = 129
"""
from transformers import GPT2Tokenizer
from cgt.api.entrypoint import SafeHyperbolicModel, SafeModelConfig
from cgt.distillation import GPT2TeacherWrapperV2

# ── HuggingFace retry helper ─────────────────────────────────────────────
def _hf_retry(fn, *args, max_attempts=4, base_delay=8, **kwargs):
    import time
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(*args, **kwargs)
        except (OSError, Exception) as exc:
            if attempt == max_attempts:
                raise
            wait = base_delay * (2 ** (attempt - 1))
            print(f'  ⚠️  HF download attempt {attempt} failed: {exc!s:.80}')
            print(f'  🔄 Retrying in {wait}s...')
            time.sleep(wait)

# ── Drive cache paths for GPT-2 ──────────────────────────────────────────
_GPT2_DRIVE_PATH = PAPER_ROOT / 'models' / 'gpt2_cache'
_GPT2_LOCAL_PATH = '/tmp/gpt2_cache'
(PAPER_ROOT / 'models').mkdir(parents=True, exist_ok=True)

def _load_gpt2_tokenizer():
    import shutil
    if Path(_GPT2_LOCAL_PATH).exists() and any(Path(_GPT2_LOCAL_PATH).iterdir()):
        print(f'  [Cache] Tokenizer /tmp hit')
        return GPT2Tokenizer.from_pretrained(_GPT2_LOCAL_PATH)
    if _GPT2_DRIVE_PATH.exists() and any(_GPT2_DRIVE_PATH.iterdir()):
        print(f'  [Cache] Tokenizer Drive hit → {_GPT2_DRIVE_PATH}')
        shutil.copytree(str(_GPT2_DRIVE_PATH), _GPT2_LOCAL_PATH, dirs_exist_ok=True)
        return GPT2Tokenizer.from_pretrained(_GPT2_LOCAL_PATH)
    print('  [HF] Downloading GPT-2 tokenizer...')
    tok = _hf_retry(GPT2Tokenizer.from_pretrained, 'gpt2')
    tok.save_pretrained(_GPT2_LOCAL_PATH)
    try:
        shutil.copytree(_GPT2_LOCAL_PATH, str(_GPT2_DRIVE_PATH), dirs_exist_ok=True)
        print(f'  [Cache] Tokenizer saved to Drive → {_GPT2_DRIVE_PATH}')
    except Exception as _e:
        print(f'  [Cache] Drive save skipped: {_e}')
    return tok

def _load_gpt2_teacher():
    _model_local = '/tmp/gpt2_model_cache'
    _model_drive = PAPER_ROOT / 'models' / 'gpt2_model_cache'
    # Tier 1: local /tmp (fast path, same session)
    if Path(_model_local).exists() and any(Path(_model_local).iterdir()):
        print(f'  [Cache] GPT-2 model /tmp hit')
        return GPT2TeacherWrapperV2(model_name=_model_local, device=DEVICE)
    # Tier 2: Drive direct (no copy — avoids /tmp OSError 107)
    if _model_drive.exists() and any(_model_drive.iterdir()):
        print(f'  [Cache] GPT-2 model Drive hit → {_model_drive}')
        return GPT2TeacherWrapperV2(model_name=str(_model_drive), device=DEVICE)
    # Tier 3: download + save to Drive
    print('  [HF] Downloading GPT-2 model (~500MB)...')
    teacher_obj = _hf_retry(GPT2TeacherWrapperV2, model_name='gpt2', device=DEVICE)
    try:
        _model_drive.mkdir(parents=True, exist_ok=True)
        teacher_obj.model.save_pretrained(str(_model_drive))
        print(f'  [Cache] GPT-2 model saved to Drive → {_model_drive}')
    except Exception as _e:
        print(f'  [Cache] Drive save skipped: {_e}')
    return teacher_obj

tokenizer  = _load_gpt2_tokenizer()
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = tokenizer.vocab_size
print(f'Tokenizer: vocab={VOCAB_SIZE}')

teacher = _load_gpt2_teacher()
print(f'Teacher (GPT-2-small): hidden={teacher.config.n_embd}  [frozen]')

student_cfg = SafeModelConfig(
    vocab_size       = VOCAB_SIZE,
    n_embd           = CFG['model']['n_embd'],
    n_layer          = CFG['model']['n_layer'],
    n_head           = CFG['model']['n_head'],
    n_positions      = CFG['model']['n_positions'],
    initial_curvature= CFG['model']['initial_curvature'],
    use_dynamics     = CFG['model']['use_dynamics'],
    hyperbolic_dim   = CFG['model']['hyperbolic_dim'],
    coupling_strength= CFG['dynamics']['coupling_strength'],
    dt               = CFG['dynamics']['dt'],
    num_steps        = CFG['dynamics']['num_steps'],
)
student = SafeHyperbolicModel(student_cfg).to(DEVICE)
n_params = student.num_parameters()

lm_head = student.core_model.lm_head
assert hasattr(lm_head, 'logit_scale'), 'logit_scale missing — update lm_head_v2.py'
assert hasattr(lm_head, 'ambient_dim'), 'ambient_dim missing — lm_head not patched to v3'

print(f'Student (HyDRA v2):')
print(f'  params={n_params:,}  arch={student_cfg.n_layer}L×{student_cfg.n_embd}d×{student_cfg.n_head}H')
print(f'  geometry=H^{student_cfg.n_embd}  K={student_cfg.initial_curvature}')
print(f'  lm_head=IntrinsicLorentz  ambient_dim={lm_head.ambient_dim}')
print(f'  logit_scale_init={lm_head.logit_scale.item():.4f}')
print('✅ MODEL SETUP')

  [Cache] Tokenizer /tmp hit
Tokenizer: vocab=50257
  [Cache] GPT-2 model Drive hit → /content/drive/MyDrive/HydraPaper_VariantF/models/gpt2_model_cache


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Teacher (GPT-2-small): hidden=768  [frozen]
Student (HyDRA v2):
  params=810,193  arch=4L×128d×4H
  geometry=H^128  K=1.0
  lm_head=IntrinsicLorentz  ambient_dim=129
  logit_scale_init=0.0884
✅ MODEL SETUP


In [5]:
"""
Cell 5 — WikiText-2 Dataset
============================
Loads WikiText-2 (raw, v1) via the Hugging Face `datasets` library and
constructs sliding-window DataLoaders.

Parameters
----------
seq_len   : 128  tokens per sample
overlap   : 64   stride of the sliding window (50% overlap)
batch_size: 16   training batch size

The sliding window avoids hard truncation at document boundaries,
ensuring the model sees coherent context across the full vocabulary.
A small 10-sentence corpus was used in earlier versions and caused
full memorization (val_ppl → 1.05); WikiText-2 has ~2M tokens and
provides genuine generalization pressure.
"""
from cgt.distillation.dataset_v2 import build_wikitext_loaders

train_loader, val_loader = build_wikitext_loaders(
    tokenizer,
    seq_len    = CFG['model']['n_positions'],
    batch_size = CFG['training']['batch_size'],
    overlap    = CFG['model']['n_positions'] // 2,
)
print(f'✅ Dataset loaded  train={len(train_loader)} batches  val={len(val_loader)} batches')
print(f'   seq_len={CFG["model"]["n_positions"]}  batch={CFG["training"]["batch_size"]}')

Loading WikiText-2 train split...
  [HF] Loading wikitext-2-raw-v1 split='train' (attempt 1/3)


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

  [HF] OK — 10,843,853 chars


Token indices sequence length is longer than the specified maximum sequence length for this model (2369968 > 1024). Running this sequence through the model will result in indexing errors


  WikiText-2 [train]: 37,029 windows (seq_len=128, overlap=64)
Loading WikiText-2 validation split...
  [HF] Loading wikitext-2-raw-v1 split='validation' (attempt 1/3)
  [HF] OK — 1,137,154 chars
  WikiText-2 [validation]: 3,828 windows (seq_len=128, overlap=64)

✅ Dataset ready:
   train: 37,029 windows | val: 3,828 windows
   seq_len=128  batch_size=16  overlap=64
✅ Dataset loaded  train=2315 batches  val=240 batches
   seq_len=128  batch=16


In [6]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️  No GPU — training will be slow (~2-3h per variant on CPU)")


DEVICE: cuda
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


## V5-A and V5-C: Corrected Runs

Both variants failed previously because `radial_proj_min_r=0.5` was not wired through.
This is now fixed in `src_FULL_V5.zip` via `DistillationConfigV2.radial_proj_min_r`
and `RiemannianAdamW.radial_proj_min_r`.

**The pre-flight check `[preflight] r_min guard: 0.5 ✅` is the critical gate.**
If it shows `N/A` or `❌`, stop and re-upload the correct src zip.

In [7]:
"""
Cell V5-AC — Corrected V5-A and V5-C Experiments
==================================================
Runs V5-A and V5-C with the radial_proj_min_r=0.5 guard active.

Previous failure mode (both runs):
  rad=0.0 from step 1 — embeddings pinned at origin.
  Cause: radial_proj_min_r was not passed through to RiemannianAdamW.
  Fix: radial_proj_min_r=0.5 is now a field of DistillationConfigV2
       and is explicitly passed to RiemannianAdamW.__init__.

Expected outcomes (with fix active):
  V5-A (Ch1 only): rdc* ~7-9  (optimizer fixed, head still drifts)
                OR rdc* ~1-3  (Ch1 was also significant)
  V5-C (both):     rdc* <1    (both channels closed — DegEq eliminated)
             For comparison: V5-B alone achieved rdc*=0.96

Reference:
  V5-D (no fix):   rdc*=10.74  PPL=913.9  (baseline)
  V5-B (Ch2 only): rdc*=0.96   PPL=576.6  (Ch2 dominant — 91% reduction)
  V5-A (broken):   RadiusCollapse
  V5-C (broken):   RadiusCollapse
"""

import copy, time, csv, torch, sys
from pathlib import Path

# Force unbuffered stdout so step logs appear in real time
import functools as _ft
_orig_print = print
print = _ft.partial(_orig_print, flush=True)
from cgt.distillation import DistillationConfigV2, DistillationTrainerV2
from cgt.api.entrypoint import SafeHyperbolicModel, SafeModelConfig

# ── Config ────────────────────────────────────────────────────────────────────
VARIANTS_TO_RUN   = ["A", "C"]     # corrected runs only
MAX_STEPS         = 3000
RUN_SEED          = 42
RADIAL_PROJ_RMIN  = 0.5            # THE critical fix

_CH = {
    "A": (True,  False),   # Ch1 only  — RiemannianAdamW + r_min guard
    "C": (True,  True),    # both      — Ch1 + Ch2
}

def _set_seed(seed):
    import random, numpy as np, os
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _flush_csv(trainer_v, log_dir):
    trn_path = log_dir / "training_metrics.csv"
    val_path = log_dir / "val_metrics.csv"
    with open(trn_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["step","train_loss","train_ppl","logit_std","diversity",
                    "ctr","lr","l_hidden","l_radius","w_std","w_entropy","rdc_proxy"])
        for r in getattr(trainer_v, "train_hist", []):
            w.writerow([r.get("step",""), f"{r.get('loss',0):.4f}",
                f"{r.get('ppl',0):.1f}", f"{r.get('logit_std',0):.4f}",
                f"{r.get('diversity',0):.2f}", f"{r.get('l_contrast',0):.3f}",
                f"{r.get('lr',0):.2e}", f"{r.get('l_hidden',0):.4f}",
                f"{r.get('l_radius',0):.4f}", f"{r.get('w_std',0):.4f}",
                f"{r.get('w_entropy',0):.4f}", f"{r.get('rdc_proxy',0):.2f}"])
    with open(val_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["step","val_loss","val_ppl","ema_slow","ema_fast","noise",
                    "phase","sg","regime_status","regime_reason","regime_code",
                    "auto_tuned","rdc_ema","regime"])
        for r in getattr(trainer_v, "val_hist", []):
            w.writerow([r.get("step",""), f"{r.get('val_loss',0):.4f}",
                f"{r.get('val_ppl',0):.1f}", f"{r.get('ema_slow',0):.4f}",
                f"{r.get('ema_fast',0):.4f}", f"{r.get('noise',0):.4f}",
                r.get("phase",1), r.get("sg",0),
                r.get("regime_status",""), r.get("regime_reason",""),
                r.get("regime_code",""),
                "1" if r.get("auto_tune_log") else "0",
                r.get("rdc_ema",""), r.get("regime","")])
    print(f"  CSVs → {trn_path.name}, {val_path.name}")

vac_results = {}

for VARIANT in VARIANTS_TO_RUN:
    ch1, ch2 = _CH[VARIANT]
    run_tag  = f"paper2_v5_{VARIANT.lower()}_corrected"
    _set_seed(RUN_SEED)

    print(f"\n{'='*64}")
    print(f"  V5-{VARIANT} CORRECTED  Ch1={ch1}  Ch2={ch2}  r_min={RADIAL_PROJ_RMIN}")
    print(f"  RUN_TAG = {run_tag}")
    print(f"{'='*64}")

    _log_dir  = PAPER_ROOT / "logs"        / run_tag
    _ckpt_dir = PAPER_ROOT / "checkpoints" / run_tag
    _log_dir.mkdir(parents=True, exist_ok=True)
    _ckpt_dir.mkdir(parents=True, exist_ok=True)
    print(f"  Run outputs -> {_log_dir}")

    cfg_v = copy.deepcopy(CFG)
    cfg_v["training"].update({
        "use_projective_kl":           False,
        "use_decoupled_ra":            False,
        "use_oted":                    True,
        "oted_r_target":               1.5,
        "lambda_radius":               0.05,
        "radial_momentum_projection":  ch1,
        "radial_proj_min_r":           RADIAL_PROJ_RMIN,  # THE FIX
        "use_angular_head":            ch2,
        "resume_from_checkpoint":      False,
        "max_steps":                   MAX_STEPS,
    })

    dist_cfg_v = DistillationConfigV2(
        alpha                    = cfg_v["training"]["alpha"],
        temperature              = cfg_v["training"]["temperature"],
        lambda_distill           = cfg_v["training"]["lambda_distill"],
        lambda_hidden            = cfg_v["training"]["lambda_hidden"],
        lambda_radius            = cfg_v["training"]["lambda_radius"],
        lambda_contrast          = cfg_v["training"]["lambda_contrast"],
        label_smoothing          = cfg_v["training"]["label_smoothing"],
        learning_rate            = cfg_v["training"]["learning_rate"],
        weight_decay             = cfg_v["training"]["weight_decay"],
        max_steps                = cfg_v["training"]["max_steps"],
        riemannian_correct_vocab   = cfg_v["training"].get("riemannian_correct_vocab",   True),
        riemannian_correct_embed   = cfg_v["training"].get("riemannian_correct_embed",   True),
        riemannian_correct_encoder = cfg_v["training"].get("riemannian_correct_encoder", True),
        use_projective_kl        = False,
        use_decoupled_ra         = False,
        use_oted                 = True,
        oted_r_target            = 1.5,
        radial_momentum_projection = ch1,
        radial_proj_min_r        = RADIAL_PROJ_RMIN,   # THE FIX — wired through
        use_angular_head         = ch2,
        warmup_steps             = cfg_v["training"]["warmup_steps"],
        gradient_clip            = cfg_v["training"]["gradient_clip"],
        eval_every               = cfg_v["training"]["eval_every"],
        log_every                = cfg_v["training"]["log_every"],
        checkpoint_every         = cfg_v["training"]["checkpoint_every"],
        lr_floor                 = cfg_v["training"]["lr_floor"],
        teacher_hidden_dim       = cfg_v["training"]["teacher_hidden_dim"],
        adaptive_lambda          = True,
        early_stopping_patience  = cfg_v["early_stopping"]["patience"],
        early_stopping_min_delta = cfg_v["early_stopping"]["min_delta"],
        ema_beta                 = cfg_v["early_stopping"]["ema_beta"],
        ema_beta_fast            = cfg_v["early_stopping"]["ema_beta_fast"],
        min_steps_before_stopping= cfg_v["early_stopping"]["min_steps"],
        trend_window             = cfg_v["early_stopping"]["window_size"],
        noise_mult               = cfg_v["early_stopping"]["noise_mult"],
        plateau_threshold        = cfg_v["early_stopping"]["plateau_threshold"],
        logit_std_min_delta      = cfg_v["early_stopping"]["logit_std_delta"],
        keep_last_n_checkpoints  = 2,
    )

    _set_seed(RUN_SEED)
    _model_cfg = SafeModelConfig(
        vocab_size        = VOCAB_SIZE,
        n_embd            = cfg_v["model"]["n_embd"],
        n_layer           = cfg_v["model"]["n_layer"],
        n_head            = cfg_v["model"]["n_head"],
        n_positions       = cfg_v["model"]["n_positions"],
        initial_curvature = cfg_v["model"]["initial_curvature"],
        use_dynamics      = cfg_v["model"]["use_dynamics"],
        hyperbolic_dim    = cfg_v["model"]["hyperbolic_dim"],
        coupling_strength = cfg_v["dynamics"]["coupling_strength"],
        dt                = cfg_v["dynamics"]["dt"],
        num_steps         = cfg_v["dynamics"]["num_steps"],
    )
    student_v = SafeHyperbolicModel(_model_cfg).to(DEVICE)

    trainer_v = DistillationTrainerV2(
        student        = student_v,
        teacher        = teacher,
        config         = dist_cfg_v,
        tokenizer      = tokenizer,
        checkpoint_dir = _ckpt_dir,
        device         = DEVICE,
    )

    # ── Pre-flight — CRITICAL: r_min guard must be 0.5 ───────────────────────
    _opt_type  = type(trainer_v.optimizer).__name__
    _rmp       = getattr(trainer_v.optimizer, "radial_momentum_projection", False)
    _r_min     = getattr(trainer_v.optimizer, "radial_proj_min_r", None)
    _parent    = getattr(trainer_v.student, "core_model", trainer_v.student)
    _head_type = type(getattr(_parent, "lm_head", None)).__name__
    _r_target  = getattr(trainer_v.config, "oted_r_target", None)
    _lam_r     = getattr(trainer_v.config, "lambda_radius", 0)
    _cfg_rmin  = getattr(trainer_v.config, "radial_proj_min_r", None)

    _ch1_ok   = (_opt_type == "RiemannianAdamW" and _rmp and _r_min == RADIAL_PROJ_RMIN) if ch1 else True
    _ch2_ok   = (_head_type == "AngularLMHead")  if ch2 else True
    _anc_ok   = (_r_target == 1.5 and _lam_r > 0)
    _rmin_ok  = (_r_min == RADIAL_PROJ_RMIN) if ch1 else True
    _cfgrm_ok = (_cfg_rmin == RADIAL_PROJ_RMIN) if ch1 else True

    print(f"  [preflight] optimizer   : {_opt_type}  rmp={_rmp}  {'✅' if _ch1_ok else '❌ Ch1 broken'}")
    print(f"  [preflight] lm_head     : {_head_type}  {'✅' if _ch2_ok else '❌ Ch2 broken'}")
    print(f"  [preflight] anchor      : r_target={_r_target} lambda_r={_lam_r}  {'✅' if _anc_ok else '❌'}")
    print(f"  [preflight] r_min guard : optimizer={_r_min}  config={_cfg_rmin}  {'✅' if (_rmin_ok and _cfgrm_ok) else '❌ RADIUS COLLAPSE RISK — STOP AND RE-UPLOAD SRC'}")

    if not (_ch1_ok and _ch2_ok and _anc_ok and _rmin_ok and _cfgrm_ok):
        print(f"  ❌ PREFLIGHT FAILED for V5-{VARIANT} — r_min guard is not active")
        print(f"     Expected: optimizer.radial_proj_min_r={RADIAL_PROJ_RMIN}")
        print(f"     Got:      optimizer.radial_proj_min_r={_r_min}")
        print(f"     Fix: upload src_FULL_V5.zip with the r_min fix and re-run Cell 2")
        vac_results[VARIANT] = {"error": "preflight_failed", "r_min_got": str(_r_min)}
        continue

    print(f"  ✅ All pre-flight checks passed — starting training")

    # Wire incremental CSV flush so trainer saves data during training
    # (protects against session disconnect before training completes)
    trainer_v._flush_csv = lambda th, vh: _flush_csv(trainer_v, _log_dir)

    # ── RadiusCollapse auto-abort (handled natively in DistillationTrainerV2) ──
    # config.radius_collapse_abort=True (default) detects rad < 0.01 × 3 steps
    # and raises RadiusCollapseAbort. CSV is flushed before abort.
    # Catch it here to continue to next variant automatically.
    from cgt.distillation import RadiusCollapseAbort
    _radius_collapse_detected = False

    t0 = time.time()
    try:
        train_hist_v, val_hist_v = trainer_v.train(train_loader, val_loader)
    except RadiusCollapseAbort as _rca:
        print(f"\n  ⚡ AUTO-ABORT: {_rca}")
        print(f"  → Continuing to next variant automatically...")
        train_hist_v = getattr(trainer_v, "train_hist", [])
        val_hist_v   = getattr(trainer_v, "val_hist",  [])
        _radius_collapse_detected = True
    elapsed = time.time() - t0

    _flush_csv(trainer_v, _log_dir)

    # ── Save best checkpoint + student weights ───────────────────────────────
    # Keeps:   distill_v2_best.pt      (best val_loss checkpoint)
    #          student_v2_weights.pt   (final student state_dict)
    # Removes: distill_v2_latest.pt    (resume checkpoint — not needed post-run)
    #          distill_v2_step_*.pt    (intermediate step checkpoints)
    # Original broken runs (paper2_v5_a, paper2_v5_c) are PRESERVED — they
    # document the RadiusCollapse mechanism used in the paper.
    try:
        trainer_v.save(is_best=True)
        _student_path = _ckpt_dir / "student_v2_weights.pt"
        torch.save(student_v.state_dict(), _student_path)
        print(f"  💾 Best ckpt  → {_ckpt_dir} / distill_v2_best.pt")
        print(f"  💾 Weights    → {_student_path}")
    except Exception as _se:
        print(f"  ⚠️  Save failed (non-critical): {_se}")

    # ── Prune intermediate checkpoints (free Drive space) ────────────────────
    try:
        import glob
        _to_delete = (
            list(_ckpt_dir.glob("distill_v2_latest.pt")) +
            list(_ckpt_dir.glob("distill_v2_step_*.pt")) +
            list(_ckpt_dir.glob("distill_v2_checkpoint_*.pt"))
        )
        for _f in _to_delete:
            _f.unlink()
            print(f"  🗑️  Pruned: {_f.name}")
        if not _to_delete:
            print(f"  ✅ No intermediate checkpoints to prune")
        # Verify what remains
        _kept = sorted(_ckpt_dir.glob("*.pt"))
        print(f"  📁 Kept in {_ckpt_dir.name}/: {[f.name for f in _kept]}")
    except Exception as _pe:
        print(f"  ⚠️  Prune failed (non-critical): {_pe}")

    # ── Collect results ───────────────────────────────────────────────────────
    _rdc_star = None
    if getattr(trainer_v, "val_hist", None):
        _rv = [float(r["rdc_ema"]) for r in trainer_v.val_hist if r.get("rdc_ema") not in (None, "")]
        _rdc_star = round(sum(_rv[-5:]) / min(5, len(_rv)), 2) if _rv else None

    _best_ppl = None
    if getattr(trainer_v, "val_hist", None):
        _pp = [float(r["val_ppl"]) for r in trainer_v.val_hist if r.get("val_ppl")]
        _best_ppl = round(min(_pp), 1) if _pp else None

    # ── Krioukov/Khrulkov diagnostic ─────────────────────────────────────────
    try:
        from cgt.diagnostics import DegEqDiagnostics, build_token_counts
        _tc  = build_token_counts(train_loader, VOCAB_SIZE, max_batches=50)
        _dg  = DegEqDiagnostics(trainer_v, tokenizer, token_counts=_tc, device=DEVICE)
        _rep = _dg.run(verbose=True)
        _diag = {
            "rho_freq_r":  _rep.rho_freq_radius,
            "k_eq":        _rep.k_equilibrium,
            "degeq_risk":  _rep.degeq_risk,
        }
    except Exception as _de:
        _diag = {}
        print(f"  [Krioukov] diagnostic skipped: {_de}")

    # Determine collapse type for paper record
    _collapse_type = "none"
    if _radius_collapse_detected:
        _collapse_type = "RadiusCollapse"
    elif _best_ppl is not None and _best_ppl >= 50000:
        _collapse_type = "DivergenceCollapse"

    vac_results[VARIANT] = {
        "ch1": ch1, "ch2": ch2,
        "rdc_star":      _rdc_star    if _rdc_star   is not None else "n/a",
        "best_ppl":      _best_ppl    if _best_ppl   is not None else "n/a",
        "elapsed_min":   round(elapsed / 60, 1),
        "rho_freq_r":    _diag.get("rho_freq_r",  "n/a"),
        "k_eq":          _diag.get("k_eq",         "n/a"),
        "degeq_risk":    _diag.get("degeq_risk",   "n/a"),
        "collapse_type": _collapse_type,
        "steps_run":     len(getattr(trainer_v, "train_hist", [])),
    }
    print(f"  → rdc*={_rdc_star}  best_ppl={_best_ppl}  ({elapsed/60:.1f} min)")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*76)
print("  V5-A and V5-C CORRECTED RESULTS")
print("="*76)
print(f"  {'Var':<4} {'Ch1':>4} {'Ch2':>4} {'rdc*':>7} {'best_ppl':>10} {'rho(f,r)':>9} {'K*':>8} {'risk':>6} {'min':>5}")
print(f"  {'-'*4} {'-'*4} {'-'*4} {'-'*7} {'-'*10} {'-'*9} {'-'*8} {'-'*6} {'-'*5}")
for v in ["A", "C"]:
    r = vac_results.get(v, {"error": "not run"})
    if "error" in r:
        print(f"  {v:<4}   PREFLIGHT FAILED  ({r.get('error')}  r_min_got={r.get('r_min_got','?')})")
    else:
        c1 = "✅" if r["ch1"] else "❌"
        c2 = "✅" if r["ch2"] else "❌"
        print(f"  {v:<4} {c1:>4} {c2:>4} "
              f"{str(r.get('rdc_star','n/a')):>7} "
              f"{str(r.get('best_ppl','n/a')):>10} "
              f"{str(r.get('rho_freq_r','n/a')):>9} "
              f"{str(r.get('k_eq','n/a')):>8} "
              f"{str(r.get('degeq_risk','n/a')):>6} "
              f"{str(r.get('elapsed_min','n/a')):>5}  "
              f"[{r.get('collapse_type','none')} @{r.get('steps_run','?')}steps]")
print()
print("  Complete V5 matrix (all variants):")
print("  V5-D  ❌ ❌  10.74      913.9  OTED baseline (no fix)")
# Dynamic V5-A/C result lines using actual run data
_a = vac_results.get("A", {})
_c = vac_results.get("C", {})
_a_line = f"{_a.get('rdc_star','?'):>6}  {_a.get('best_ppl','?'):>9}  "\
          f"{_a.get('collapse_type','?')!s} @{_a.get('steps_run','?')!s}steps"
_c_line = f"{_c.get('rdc_star','?'):>6}  {_c.get('best_ppl','?'):>9}  "\
          f"{_c.get('collapse_type','?')!s} @{_c.get('steps_run','?')!s}steps"
print(f"  V5-A  ✅ ❌  {_a_line}")
print("  V5-B  ❌ ✅   0.96      576.6  none  — Ch2 alone, 91% rdc* reduction ★")
print(f"  V5-C  ✅ ✅  {_c_line}")
print()
print("  ── Collapse detection (auto-abort) ──────────────────────────────────────")
print("  rad<0.01 × 3 consecutive log steps → auto-abort, CSV flushed, next variant")
print("  V5-A RadiusCollapse = Ch1 isolado instável sem Ch2 (optimizer não estabiliza)")
print("  V5-C RadiusCollapse = Ch1 destrói a estabilidade que Ch2 forneceria")
print("  V5-C sem colapso    = ambos os canais são necessários → paper fechado ✅")
print()
print("  Interpretation — definitive:")
print("  rdc*(A) << 10.74   → Ch1 (optimizer momentum) is a real source")
print("  rdc*(A) >> rdc*(B) → Ch2 is more dominant than Ch1")
print("  rdc*(C) < rdc*(B)  → both channels together stronger than Ch2 alone")
print("  rdc*(C) ≈ rdc*(B)  → Ch1 adds negligible effect over Ch2")
print("  rdc*(C) ≈ 0        → DegEq fully eliminated — paper closed ✅")



  V5-A CORRECTED  Ch1=True  Ch2=False  r_min=0.5
  RUN_TAG = paper2_v5_a_corrected
  Run outputs -> /content/drive/MyDrive/HydraPaper_VariantF/logs/paper2_v5_a_corrected
  [preflight] optimizer   : RiemannianAdamW  rmp=True  ✅
  [preflight] lm_head     : HyperbolicLMHeadV2  ✅
  [preflight] anchor      : r_target=1.5 lambda_r=0.05  ✅
  [preflight] r_min guard : optimizer=0.5  config=0.5  ✅
  ✅ All pre-flight checks passed — starting training

🎓 STARTING v2 TRAINING
Teacher:        ON
Lambda distill: 0.3
Max steps:      3000


/tmp/ipykernel_13430/1086188806.py:174: UserWarning: [V5] RiemannianAdamW with radial_momentum_projection=True active.
  trainer_v = DistillationTrainerV2(
/tmp/ipykernel_13430/1086188806.py:174: UserWarning: [Paper2] OTED active: Origin-Tangent Euclidean Distillation.
  trainer_v = DistillationTrainerV2(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1787: RuntimeWarning: [RadiusCollapse] mean_radius=0.0014 < 0.1. Embeddings collapsed to origin.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1787: RuntimeWarning: [TangentCollapse] mean tangent norm=0.00e+00 < 1e-6. Student embeddings have collapsed to manifold origin. Cosine/contrastive gradients are zero — training is stuck. Increase lambda_radius or reduce lambda_distill.
  return forward_call(*args, **kwargs)
/content/src/src/cgt/distillation/distillation_v2.py:2962: RuntimeWarning: [TangentCollapse] mean tangent norm=0.00e+00 < 1e-6. Student embeddings 

  step=    50  loss=10.7188  ppl=50257.0  lr=5.00e-05  logit_std=0.0000❌  div=1.00✅  hid=1.000  rad=0.001❌  ctr=2.773  rdc=0.0✅  w_std=0.0000✅  w_ent=4.85✅
  step=   100  loss=10.7127  ppl=50257.0  lr=1.00e-04  logit_std=0.0000❌  div=0.99✅  hid=1.000  rad=0.001❌  ctr=2.773  rdc=0.0✅  w_std=0.0000✅  w_ent=4.85✅
  CSVs → training_metrics.csv, val_metrics.csv

  ⚡ AUTO-ABORT: RadiusCollapse at step 150: mean_rad=0.0014 < 0.01 for 3 consecutive log steps. Auto-aborted — continuing to next experiment.
  → Continuing to next variant automatically...
  CSVs → training_metrics.csv, val_metrics.csv
💾 Best v2 model saved!
  💾 Best ckpt  → /content/drive/MyDrive/HydraPaper_VariantF/checkpoints/paper2_v5_a_corrected / distill_v2_best.pt
  💾 Weights    → /content/drive/MyDrive/HydraPaper_VariantF/checkpoints/paper2_v5_a_corrected/student_v2_weights.pt
  🗑️  Pruned: distill_v2_latest.pt
  📁 Kept in paper2_v5_a_corrected/: ['distill_v2_best.pt', 'distill_v2_ckpt_150.pt', 'distill_v2_ckpt_200.pt', 'di

/tmp/ipykernel_13430/1086188806.py:174: UserWarning: [V5] AngularLMHead active — dL/dr=0 for all k.
  trainer_v = DistillationTrainerV2(


  step=    50  loss=nan  ppl=50257.0  lr=5.00e-05  logit_std=0.0000❌  div=1.00✅  hid=nan  rad=0.001❌  ctr=0.000  rdc=nan❌  w_std=0.0000✅  w_ent=4.85✅
  step=   100  loss=nan  ppl=50257.0  lr=1.00e-04  logit_std=0.0000❌  div=0.99✅  hid=nan  rad=0.001❌  ctr=0.000  rdc=nan❌  w_std=0.0000✅  w_ent=4.85✅
  CSVs → training_metrics.csv, val_metrics.csv

  ⚡ AUTO-ABORT: RadiusCollapse at step 150: mean_rad=0.0014 < 0.01 for 3 consecutive log steps. Auto-aborted — continuing to next experiment.
  → Continuing to next variant automatically...
  CSVs → training_metrics.csv, val_metrics.csv
💾 Best v2 model saved!
  💾 Best ckpt  → /content/drive/MyDrive/HydraPaper_VariantF/checkpoints/paper2_v5_c_corrected / distill_v2_best.pt
  💾 Weights    → /content/drive/MyDrive/HydraPaper_VariantF/checkpoints/paper2_v5_c_corrected/student_v2_weights.pt
  🗑️  Pruned: distill_v2_latest.pt
  📁 Kept in paper2_v5_c_corrected/: ['distill_v2_best.pt', 'distill_v2_ckpt_150.pt', 'student_v2_weights.pt']
  [Krioukov] dia

## Optional: Prune Intermediate Checkpoints

Run **after** all training is complete. Frees Drive space by removing
intermediate step checkpoints while preserving the essential files.

**The broken V5-A and V5-C original runs are never deleted** —
they document the RadiusCollapse mechanism used in the paper.


In [8]:
"""
Optional: Prune intermediate checkpoints from ALL completed runs.

Keeps for each run:
  ✅ distill_v2_best.pt       — best validation checkpoint
  ✅ student_v2_weights.pt    — final student state dict

Removes (intermediate, only needed for resume):
  🗑️  distill_v2_latest.pt
  🗑️  distill_v2_step_*.pt
  🗑️  distill_v2_checkpoint_*.pt

IMPORTANT: Original broken runs (paper2_v5_a, paper2_v5_c) are
PRESERVED in full — they document RadiusCollapse for the paper.
Only intermediate checkpoints are removed.

Run this cell ONLY after confirming all training is complete
and all student_v2_weights.pt files have been saved.
"""

import os
from pathlib import Path

CKPT_ROOT = PAPER_ROOT / "checkpoints"

# Runs to preserve in full (broken — scientific evidence)
PRESERVE_FULL = {
    "paper2_v5_a",    # RadiusCollapse original — used in paper
    "paper2_v5_c",    # RadiusCollapse original — used in paper
}

total_freed = 0
print("=" * 60)
print("  CHECKPOINT PRUNING REPORT")
print("=" * 60)

for run_dir in sorted(CKPT_ROOT.iterdir()):
    if not run_dir.is_dir():
        continue

    run_name = run_dir.name
    all_pts = sorted(run_dir.glob("*.pt"))

    if run_name in PRESERVE_FULL:
        print(f"\n  {run_name}/  [PRESERVED — scientific record]")
        for f in all_pts:
            print(f"    ✅ {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
        continue

    # Prune intermediates, keep best + weights
    keep = {"distill_v2_best.pt", "student_v2_weights.pt"}
    print(f"\n  {run_name}/")
    for f in all_pts:
        if f.name in keep:
            print(f"    ✅ KEEP   {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
        else:
            freed = f.stat().st_size
            f.unlink()
            total_freed += freed
            print(f"    🗑️  DEL    {f.name}  ({freed/1e6:.1f} MB freed)")

print(f"\n  Total freed: {total_freed/1e6:.1f} MB")
print("=" * 60)
print("  Kept for all runs: distill_v2_best.pt + student_v2_weights.pt")
print("  Preserved intact:  paper2_v5_a, paper2_v5_c (RadiusCollapse evidence)")


  CHECKPOINT PRUNING REPORT

  paper2_oted/
    ✅ KEEP   distill_v2_best.pt  (88.2 MB)

  paper2_v5_a/  [PRESERVED — scientific record]
    ✅ distill_v2_best.pt  (88.3 MB)
    ✅ distill_v2_ckpt_3000.pt  (88.3 MB)
    ✅ distill_v2_latest.pt  (88.3 MB)
    ✅ student_v2_weights.pt  (29.1 MB)

  paper2_v5_a_corrected/
    ✅ KEEP   distill_v2_best.pt  (88.1 MB)
    🗑️  DEL    distill_v2_ckpt_150.pt  (88.1 MB freed)
    🗑️  DEL    distill_v2_ckpt_200.pt  (88.1 MB freed)
    🗑️  DEL    distill_v2_ckpt_500.pt  (88.1 MB freed)
    ✅ KEEP   student_v2_weights.pt  (29.1 MB)

  paper2_v5_b/
    ✅ KEEP   distill_v2_best.pt  (164.5 MB)
    ✅ KEEP   student_v2_weights.pt  (54.7 MB)

  paper2_v5_c/  [PRESERVED — scientific record]
    ✅ distill_v2_best.pt  (164.5 MB)
    ✅ distill_v2_ckpt_3000.pt  (164.5 MB)
    ✅ distill_v2_latest.pt  (164.5 MB)
    ✅ student_v2_weights.pt  (54.7 MB)

  paper2_v5_c_corrected/
    ✅ KEEP   distill_v2_best.pt  (164.3 MB)
    🗑️  DEL    distill_v2_ckpt_150.pt  (164.3 MB